# Synthetic transaction data generation

In this notebook I'll create reproducible synthetic customer, account, merchant and transaction data using numpy

## Objectives

- Generate behavioural profiles for synthetic customers
- Create related account and merchant tables
- Generate transactions using vectorised NumPy operations
- Model customer specific amounts, hours, categories and international activity
- Inject known behavioural anomalies
- Preserve anomaly ground truth for later evaluation
- Prepare scalable dataset sizes for future performance testing

Data quality errors and behavioural anomalies are different, as data quality errors are invalid or corrupted records while behavioural anomalies are valid records that are unusual for a customer

In [1]:
from pathlib import Path 
import numpy as np 
import pandas as pd

In [2]:
current_directory = Path.cwd()

if current_directory.name == "notebooks":
    project_root = current_directory.parent
else:
    project_root = current_directory

synthetic_directory = (
    project_root / "data" / "synthetic" / "sample"
)

synthetic_directory.mkdir(parents=True, exist_ok=True)

In [3]:
SEED = 42

rng = np.random.default_rng(SEED)

In [4]:
dataset_sizes = {"debug": 100,
                 "development": 1_000,
                 "integration": 100_000,
                 "benchmark": 1_000_000
                 }

customer_counts = {
    "debug": 20,
    "development": 100,
    "integration": 5_000,
    "benchmark": 50_000,
}

DATASET_TIER = "development"

N_TRANSACTIONS = dataset_sizes[DATASET_TIER]

N_CUSTOMERS = customer_counts[DATASET_TIER]

In [5]:
# I will use arrays as catalogues

countries = np.array(
    [
        "Spain",
        "France",
        "Germany",
        "Italy",
        "Portugal",
        "Ireland",
        "United Kingdom",
        "United States",
    ]
)

currencies = np.array(
    [
        "EUR",
        "EUR",
        "EUR",
        "EUR",
        "EUR",
        "EUR",
        "GBP",
        "USD",
    ]
)

segments = np.array(
    [
        "Student",
        "Standard",
        "Premium",
    ]
)

categories = np.array(
    [
        "Groceries",
        "Restaurants",
        "Books",
        "Electronics",
        "Sports",
        "Travel",
        "Home",
        "Entertainment",
        "Fashion",
        "Health",
    ]
)

n_countries = len(countries)
n_categories = len(categories)

In [6]:
# Ids, country and segment

customer_ids = np.array([f"C{i:05d}" for i in range(1, N_CUSTOMERS + 1)])

customer_ids[:5]

array(['C00001', 'C00002', 'C00003', 'C00004', 'C00005'], dtype='<U6')

In [7]:
customer_country_idx = rng.integers(0, n_countries, size = N_CUSTOMERS)

In [8]:
customer_country_idx[:5]

array([0, 6, 5, 3, 3])

In [9]:
countries[customer_country_idx[:5]]

array(['Spain', 'United Kingdom', 'Ireland', 'Italy', 'Italy'],
      dtype='<U14')

In [10]:
customer_segment_idx = rng.choice(
    len(segments),
    size = N_CUSTOMERS,
    p = [0.20, 0.55, 0.25]
)

In [11]:
segments[customer_segment_idx[:10]]

array(['Student', 'Student', 'Premium', 'Standard', 'Standard', 'Premium',
       'Standard', 'Standard', 'Student', 'Student'], dtype='<U8')

In [12]:
age_means = np.array([23, 40, 52])
# I choose this stds becase students would be more concentrated around 23 years old while standard and premium
# segments would have more variaty
age_stds = np.array([3, 10, 10])

In [13]:
ages = np.clip(
    np.rint(
        rng.normal(
            loc = age_means[customer_segment_idx],
            scale = age_stds[customer_segment_idx]
        )
    ).astype(int),
    18,
    80
)

In [14]:
print(ages.min())
print(ages.max())

19
66


In [15]:
# I  create a different standard amount for each segment, customer
segment_base_amount = np.array([
    25.0, 60.0, 120.0
])

customer_base_amount = (
    segment_base_amount[customer_segment_idx]
    * rng.lognormal(mean = 0, sigma = 0.25, size = N_CUSTOMERS)
)

customer_base_amount[:5]

array([ 51.68370857,  18.65257428, 109.44580591,  65.34843226,
        92.43622741])

In [16]:
# I generate the prefered shopping hour of each customer
preferred_hours = np.mod(
        np.rint(
            rng.normal(
                loc = 15, # 15:00
                scale = 3,
                size = N_CUSTOMERS
            )
        ).astype(int),
        24
)

In [17]:
print(preferred_hours.min())
print(preferred_hours.max())

8
22


In [18]:
# Initial international prob / customer
base_international_probability = (
    np.array(
        [0.08, 0.12, 0.22] # Student, standard, premium
    )
)

In [21]:
international_probability = np.clip(
    base_international_probability[
        customer_segment_idx
    ]
    # I add an individual variation from a beta distribution
    + rng.beta( 2, 12, size=N_CUSTOMERS),
    0.02,
    0.65
)

In [22]:
# I also want some customers to appear in many transactions and vice versa, so I will use a gamma
# distribution to generate postive assymetric values which I'll turn into probabilities

# positive activity weight
activity_raw = rng.gamma(shape = 2.0, scale = 1.0, size = N_CUSTOMERS)

activity_probability = (activity_raw / activity_raw.sum())

In [24]:
activity_probability.sum() 

np.float64(1.0000000000000002)

In [25]:
# Preference vector / customer
category_preferences = rng.dirichlet(
    np.full(
        n_categories, 0.45 # < 1 for more concentrated preferences
    ),
    size = N_CUSTOMERS
)

In [26]:
category_preferences.sum(axis=1)[:5]

array([1., 1., 1., 1., 1.])

In [29]:
# I know generate how many days after 2019/01/01 each customer signed up
# I will cover 0-5 years
signup_days = rng.integers(0, 365*5, size=N_CUSTOMERS)

In [30]:
signup_dates = (pd.Timestamp("2019-01-01")
                + pd.to_timedelta(
                    signup_days, unit = "D"
                ))

In [31]:
# Customer table
customers = pd.DataFrame(
    {
        "customer_id": customer_ids,
        "age": ages,
        "country": countries[ customer_country_idx],
        "signup_date": signup_dates,
        "customer_segment": segments[customer_segment_idx],
        "typical_amount": np.round(customer_base_amount,2 ),
        "preferred_hour": (preferred_hours),
        "international_probability": (
            np.round(international_probability, 3)
        ),
        "activity_weight": np.round(activity_probability, 6)
    }
)

In [32]:
customers.head()

,customer_id,age,country,signup_date,customer_segment,typical_amount,preferred_hour,international_probability,activity_weight
0,C00001,26,Spain,2020-06-20,Student,51.68,10,0.321,0.026483
1,C00002,23,United Kingdom,2021-12-09,Student,18.65,19,0.403,0.009543
2,C00003,50,Ireland,2023-12-17,Premium,109.45,16,0.433,0.052133
3,C00004,30,Italy,2019-01-05,Standard,65.35,18,0.350,0.004717
4,C00005,23,Italy,2023-10-13,Standard,92.44,13,0.173,0.006435


In [33]:
customer_category_preferences = (pd.DataFrame(category_preferences, columns = categories))

In [34]:
customer_category_preferences.insert(0,"customer_id",customer_ids)

In [35]:
customer_category_preferences.head()

,customer_id,Groceries,Restaurants,Books,Electronics,Sports,Travel,Home,Entertainment,Fashion,Health
0,C00001,0.206261,0.120822,1.318096e-01,0.034190,0.076669,0.009607,0.026932,0.175351,0.017322,0.201037
1,C00002,0.127912,0.022040,2.636316e-02,0.381002,0.000479,0.139399,0.249938,0.001373,0.003777,0.047717
2,C00003,0.083236,0.100813,1.390525e-02,0.026400,0.078256,0.135766,0.163659,0.014220,0.146350,0.237395
3,C00004,0.004862,0.000783,1.567342e-08,0.224396,0.126763,0.005939,0.369622,0.079194,0.001587,0.186854
4,C00005,0.093042,0.092102,1.572832e-02,0.004447,0.001278,0.003067,0.135671,0.059135,0.018204,0.577326


In [39]:
# Now I will generate the accounts; I want 75% to have one account, and 25% to have two
account_counts = (1 + (rng.random(N_CUSTOMERS) < 0.25).astype(int))


In [40]:
account_customer_idx = np.repeat(np.arange(N_CUSTOMERS), account_counts)

In [42]:
N_ACCOUNTS = len(account_customer_idx)
N_ACCOUNTS

126

In [43]:
# ids for the accounts
accounts_ids = np.array([
    f"A{i:05d}" for i in range(1, N_ACCOUNTS + 1)
])

In [44]:
account_starts = np.cumsum(
    np.r_[0,account_counts[:-1]]
)

In [45]:
# position of the account
account_sequence = (
    np.arange(N_ACCOUNTS) - np.repeat(account_starts, account_counts)
)

In [46]:
# type of accounts -> 2nd account is for savings

account_types = np.where(
    account_sequence == 0,
    np.where(
        segments[customer_segment_idx[account_customer_idx]] # segment of each account's owner
    
    == "Student", 
    "Student",
    "Current"
    ),
    "Savings"
)

In [48]:
# opening date of the account

opening_dates = (
    signup_dates[account_customer_idx] + pd.to_timedelta(
        rng.integers(0, 365, size = N_ACCOUNTS),  unit = "D"
    )
)

In [49]:
accounts = pd.DataFrame(
    {
        "account_id": accounts_ids,

        "customer_id": customer_ids[account_customer_idx],

        "account_type": account_types,

        "currency": currencies[
            customer_country_idx[
                account_customer_idx
            ]
        ],
        "opening_date": opening_dates,

        "status": "Active",
    }
)

In [50]:
accounts.head()

,account_id,customer_id,account_type,currency,opening_date,status
0,A00001,C00001,Student,EUR,2021-02-10,Active
1,A00002,C00002,Student,GBP,2022-02-24,Active
2,A00003,C00002,Savings,GBP,2022-05-27,Active
3,A00004,C00003,Current,EUR,2024-02-23,Active
4,A00005,C00004,Current,EUR,2019-01-14,Active


In [51]:
# Merchants catalogue
merchant_category_values = np.repeat(categories, n_countries)
merchant_country_values = np.tile(countries, n_categories)
merchant_currency_values = np.tile(currencies, n_categories)

In [53]:
N_MERCHANTS = len(merchant_category_values)
N_MERCHANTS

80

In [54]:
merchant_ids = np.array(
    [
        f"M{i:05d}" for i in range(1, N_MERCHANTS + 1)
    ]
)

In [57]:
merchant_names = np.array(
    [        f"{category} {country} Hub"
        for category, country in zip(
            merchant_category_values,
            merchant_country_values,
        )
    ]
)
merchant_names[:5]

array(['Groceries Spain Hub', 'Groceries France Hub',
       'Groceries Germany Hub', 'Groceries Italy Hub',
       'Groceries Portugal Hub'], dtype='<U32')

In [58]:
# Now I will define the online categories 
online_categories = np.array(
    [
        "Books",
        "Electronics",
        "Travel",
        "Home",
        "Entertainment",
        "Fashion",
    ]
)

In [59]:
merchant_online = np.isin(merchant_category_values, online_categories)

In [60]:
# Merchant dataframe

merchants = pd.DataFrame(
    {
        "merchant_id": merchant_ids,
        "merchant_name": merchant_names,
        "category": (
            merchant_category_values
        ),
        "country": (
            merchant_country_values
        ),
        "currency": (
            merchant_currency_values
        ),
        "online": merchant_online,
    }
)


In [61]:
merchants.head()

,merchant_id,merchant_name,category,country,currency,online
0,M00001,Groceries Spain Hub,Groceries,Spain,EUR,False
1,M00002,Groceries France Hub,Groceries,France,EUR,False
2,M00003,Groceries Germany Hub,Groceries,Germany,EUR,False
3,M00004,Groceries Italy Hub,Groceries,Italy,EUR,False
4,M00005,Groceries Portugal Hub,Groceries,Portugal,EUR,False
